In [ ]:
# Azure AI Language Python client library: pip install azure-ai-textanalytics
from azure.core.credentials import AzureKeyCredential
from azure.ai.textanalytics import TextAnalyticsClient
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import os, time, uuid, random, re, html

In [ ]:
load_dotenv(".env")

LANGUAGE_ENDPOINT = https://REDACTED_ENDPOINT/
LANGUAGE_KEY = REDACTED_AZURE_KEY

if not LANGUAGE_ENDPOINT or not LANGUAGE_KEY:
    raise ValueError("Missing AZURE_LANGUAGE_ENDPOINT or AZURE_LANGUAGE_KEY in .env")

client = TextAnalyticsClient(
    endpoint=LANGUAGE_ENDPOINT,
    credential=AzureKeyCredential(LANGUAGE_KEY)
)

print("Azure Language client ready.")

In [ ]:
# =========================
# SOURCE FOLDER
# =========================
SOURCE_DIR = Path(r"C:\Users\Edwin\Desktop\Lithan Academy\ModellingProject\datasets\abc_bank_reviews_text_analysis")

if not SOURCE_DIR.exists():
    raise FileNotFoundError(f"Folder not found: {SOURCE_DIR}")

print("Reading files from:", SOURCE_DIR)

In [ ]:
# =========================
# HELPERS
# =========================
def strip_html_tags(text: str) -> str:
    text = html.unescape(text or "")
    text = re.sub(r"<br\s*/?>", "\n", text, flags=re.IGNORECASE)
    text = re.sub(r"</p\s*>", "\n", text, flags=re.IGNORECASE)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_eml_body(eml_path: Path) -> str:
    with eml_path.open("rb") as f:
        msg = BytesParser(policy=policy.default).parse(f)

    plain_parts = []
    html_parts = []

    if msg.is_multipart():
        for part in msg.walk():
            content_type = part.get_content_type()
            disposition = str(part.get_content_disposition() or "").lower()

            if disposition == "attachment":
                continue

            try:
                content = part.get_content()
            except Exception:
                continue

            if not content:
                continue

            if content_type == "text/plain":
                plain_parts.append(str(content))
            elif content_type == "text/html":
                html_parts.append(str(content))
    else:
        try:
            content = msg.get_content()
        except Exception:
            content = ""

        if msg.get_content_type() == "text/plain":
            plain_parts.append(str(content))
        elif msg.get_content_type() == "text/html":
            html_parts.append(str(content))

    if plain_parts:
        return "\n".join(plain_parts).strip()

    if html_parts:
        return strip_html_tags("\n".join(html_parts))

    return ""


def extract_text_file(file_path: Path) -> str:
    for enc in ["utf-8", "utf-8-sig", "cp1252", "latin-1"]:
        try:
            return file_path.read_text(encoding=enc).strip()
        except Exception:
            continue
    return ""


def extract_email_text(file_path: Path) -> str:
    ext = file_path.suffix.lower()

    if ext == ".eml":
        return extract_eml_body(file_path)

    if ext in [".txt", ".log"]:
        return extract_text_file(file_path)

    return ""


def extract_key_phrases_from_text(text: str):
    result = client.extract_key_phrases([text])[0]

    if result.is_error:
        raise RuntimeError(f"Key phrase extraction failed: {result.error}")

    phrases = list(result.key_phrases)

    return {
        "key_phrases": phrases,
        "key_phrase_count": len(phrases)
    }

In [ ]:
# =========================
# PROCESS ALL EMAIL FILES
# =========================
supported_files = []
for pattern in ["*.eml", "*.txt", "*.log"]:
    supported_files.extend(SOURCE_DIR.glob(pattern))

supported_files = sorted(supported_files)

if not supported_files:
    raise FileNotFoundError("No .eml, .txt, or .log files found in the folder.")

print(f"Found {len(supported_files)} files")

results = []
phrases_output = []

for file_path in supported_files:
    print("Processing:", file_path.name)

    text = extract_email_text(file_path)

    if not text.strip():
        print(f"  Skipped - no readable text: {file_path.name}")
        continue

    try:
        analysis = extract_key_phrases_from_text(text)

        results.append({
            "file_name": file_path.name,
            "file_path": str(file_path),
            "char_count": len(text),
            "key_phrase_count": analysis["key_phrase_count"],
            "key_phrases_joined": " | ".join(analysis["key_phrases"]),
            "text_preview": text[:300]
        })

        for phrase in analysis["key_phrases"]:
            phrases_output.append({
                "file_name": file_path.name,
                "key_phrase": phrase
            })

    except Exception as e:
        results.append({
            "file_name": file_path.name,
            "file_path": str(file_path),
            "char_count": len(text),
            "key_phrase_count": None,
            "key_phrases_joined": "",
            "text_preview": str(e)
        })
        print(f"  Error: {e}")

df_keyphrases = pd.DataFrame(results)
df_keyphrase_items = pd.DataFrame(phrases_output)

print("Done.")

In [ ]:
# =========================
# VIEW RESULTS
# =========================
display(df_keyphrases)
display(df_keyphrase_items.head(50))

In [ ]:
# =========================
# TOP PHRASES
# =========================
if not df_keyphrase_items.empty:
    df_top_phrases = (
        df_keyphrase_items["key_phrase"]
        .value_counts()
        .reset_index()
    )
    df_top_phrases.columns = ["key_phrase", "count"]

    display(df_top_phrases.head(30))
else:
    df_top_phrases = pd.DataFrame(columns=["key_phrase", "count"])
    print("No key phrases extracted.")

In [ ]:
# =========================
# EXPORT TO CSV
# =========================
output_dir = SOURCE_DIR / "analysis_output"
output_dir.mkdir(exist_ok=True)

summary_csv = output_dir / "email_keyphrase_summary.csv"
items_csv = output_dir / "email_keyphrase_items.csv"
top_csv = output_dir / "email_top_keyphrases.csv"

df_keyphrases.to_csv(summary_csv, index=False, encoding="utf-8-sig")
df_keyphrase_items.to_csv(items_csv, index=False, encoding="utf-8-sig")
df_top_phrases.to_csv(top_csv, index=False, encoding="utf-8-sig")

print("Saved:")
print(summary_csv)
print(items_csv)
print(top_csv)